## LLM-as-a-Judge: avaliando uma aplicação RAG com uma métrica personalizada

RAG (Retrieval Augmented Generation) é um dos casos de uso mais populares de LLMs — e um dos mais difíceis de avaliar. As métricas comuns nem sempre se encaixam no seu caso de uso, por serem genéricas demais. Aqui, definimos uma **métrica aditiva de 3 pontos** que avalia as respostas quanto a: aderência ao contexto, completude e concisão.

*Nota: esta é uma métrica fictícia, apenas para demonstração. Defina métricas e critérios de acordo com o seu caso de uso.*

**Modelos utilizados (via Hugging Face Inference API):**

- **Modelo avaliado** (gera as respostas do "RAG"): [Qwen/Qwen3-8B](https://huggingface.co/Qwen/Qwen3-8B) (variável `HF_MODEL_ID`)
- **Modelo juiz** (avalia as respostas): [Qwen/Qwen3-14B](https://huggingface.co/Qwen/Qwen3-14B) (variável `HF_MODEL_EVAL_ID`)

In [43]:
%pip install huggingface_hub python-dotenv tqdm --quiet


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


As credenciais e os IDs dos modelos são carregados do arquivo `.env` na raiz do projeto. Usamos o cliente assíncrono da Hugging Face para pontuar várias amostras em paralelo.

In [44]:
import os
import asyncio
from pathlib import Path
from dotenv import load_dotenv
from huggingface_hub import AsyncInferenceClient
from tqdm.asyncio import tqdm_asyncio

# Carrega as variáveis de ambiente do arquivo .env na raiz do projeto
load_dotenv(dotenv_path=Path.cwd() / ".env")

HF_TOKEN = os.environ["HF_TOKEN"]
HF_MODEL_ID = os.environ["HF_MODEL_ID"]           # modelo avaliado: Qwen/Qwen3-8B
HF_MODEL_EVAL_ID = os.environ["HF_MODEL_EVAL_ID"] # modelo juiz: Qwen/Qwen3-14B

# concorrência máxima
sem = asyncio.Semaphore(5)

# Cliente assíncrono da Hugging Face Inference API
client = AsyncInferenceClient(api_key=HF_TOKEN, provider="auto")

Para avaliar as respostas, definimos os `additive_criteria`, os `evaluation_steps` e o `json_schema` do formato de saída.

In [45]:
def format_examples(examples):
    return "\n".join([
        f'Pergunta: {ex["question"]}\nContexto: {ex["context"]}\nResposta: {ex["answer"]}\nAvaliação:{ex["eval"]}' 
        for ex in examples
    ])

EVALUATION_PROMPT_TEMPLATE = """Você é um juiz especialista em avaliar aplicações de Retrieval Augmented Generation (RAG). Sua tarefa é avaliar uma resposta com base em um contexto e uma pergunta, usando os critérios abaixo.

Critérios de avaliação (pontuação aditiva, 0-3):
{additive_criteria}

Passos da avaliação:
{evaluation_steps}

Formato de saída:
{json_schema}

Exemplos:
{examples}

Agora, avalie o seguinte:

Pergunta:
{question}
Contexto:
{context}
Resposta:
{answer}
"""

ADDITIVE_CRITERIA = """1. Contexto: atribua 1 ponto se a resposta usa apenas informações fornecidas no contexto, sem introduzir detalhes externos ou inventados.
2. Completude: adicione 1 ponto se a resposta cobre todos os elementos-chave da pergunta com base no contexto disponível, sem omissões.
3. Concisão: adicione 1 ponto final se a resposta usa o mínimo de palavras possível para responder à pergunta, sem redundância."""

EVALUATION_STEPS="""1. Leia a pergunta e a resposta com atenção para entender o contexto.
2. Analise cada critério um a um e verifique se a resposta o atende.
3. Escreva seu raciocínio para cada critério, explicando por que atribuiu ou não o ponto. Seja específico e cite elementos da resposta que influenciaram sua decisão.
4. Para cada critério atendido, some o ponto correspondente (máximo de 1 ponto por critério; apenas pontos inteiros).
5. Formate a avaliação no formato de saída especificado, em JSON válido, com o campo "reasoning" para a explicação passo a passo e o campo "total_score" para o total calculado. Revise a resposta: ela precisa ser um JSON válido."""

JSON_SCHEMA="""{
  "reasoning": "Explicação passo a passo dos critérios de avaliação, por que o ponto foi atribuído ou não.",
  "total_score": soma dos pontos dos critérios,
}"""

Para melhorar a avaliação, usamos três exemplos few-shot (notas 0, 1 e 3), definidos abaixo:

In [46]:
import json

few_shot_examples = [
    {
        "question": "Qual foi a inflação medida pelo IPCA em 2023?",
        "context": "O Índice Nacional de Preços ao Consumidor Amplo (IPCA), medido pelo IBGE, fechou 2023 em 4,62%, abaixo do teto da meta de inflação.",
        "answer": "A inflação medida pelo IPCA em 2023 foi de 4,62%.",
        "eval": json.dumps({
            "reasoning": "1. Contexto: a resposta usa apenas a informação do contexto (4,62%). 1 ponto. 2. Completude: responde exatamente o valor pedido. 1 ponto. 3. Concisão: direta, sem redundância. 1 ponto.",
            "total_score": 3,
        }, ensure_ascii=False),
    },
    {
        "question": "Quantos aviões a Embraer entregou em 2023?",
        "context": "A Embraer entregou 181 aeronaves em 2023, entre jatos comerciais e executivos, um crescimento de 13% em relação às 160 entregas de 2022.",
        "answer": "A Embraer, uma das maiores fabricantes de aviões do mundo, com sede em São José dos Campos, entregou 181 aeronaves em 2023 e mantém uma carteira de pedidos bilionária.",
        "eval": json.dumps({
            "reasoning": "1. Contexto: a resposta adiciona informações externas ao contexto (sede em São José dos Campos, carteira de pedidos). 0 pontos. 2. Completude: informa o número de entregas (181). 1 ponto. 3. Concisão: resposta com informações desnecessárias. 0 pontos.",
            "total_score": 1,
        }, ensure_ascii=False),
    },
    {
        "question": "Qual foi a taxa Selic ao final de 2023?",
        "context": "Em dezembro de 2023, o Comitê de Política Monetária (Copom) do Banco Central reduziu a taxa Selic para 11,75% ao ano.",
        "answer": "A taxa Selic encerrou 2023 em 13,75% ao ano.",
        "eval": json.dumps({
            "reasoning": "1. Contexto: o valor citado (13,75%) contradiz o contexto (11,75%). 0 pontos. 2. Completude: a resposta não traz a informação correta. 0 pontos. 3. Concisão: embora curta, a resposta está incorreta e não atende ao critério. 0 pontos.",
            "total_score": 0,
        }, ensure_ascii=False),
    },
]

Em seguida, definimos o método `get_eval_score`, que monta o prompt, chama o juiz (Qwen3-14B) e extrai o JSON da resposta.

In [47]:
import json

def extract_json(text):
    # remove cercas de markdown (```json ... ```), se houver, e extrai o JSON
    text = text.strip()
    if "```" in text:
        text = text.split("```")[1]
        if text.startswith("json"):
            text = text[4:]
    start = text.find("{")
    end = text.rfind("}")
    return json.loads(text[start:end + 1])

async def get_eval_score(sample):
    prompt = EVALUATION_PROMPT_TEMPLATE.format(
        additive_criteria=ADDITIVE_CRITERIA,
        evaluation_steps=EVALUATION_STEPS,
        json_schema=JSON_SCHEMA,
        examples=format_examples(few_shot_examples),
        question=sample["question"],
        context=sample["context"],
        answer=sample["answer"]
    )
    # Remova o comentário para ver o prompt
    # print(prompt)
    response = await client.chat.completions.create(
        model=HF_MODEL_EVAL_ID,  # juiz: Qwen/Qwen3-14B
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=2048,
    )
    results = extract_json(response.choices[0].message.content or "{}")
    # Adiciona o resultado da avaliação à amostra
    return {**sample, **results}

# Método auxiliar para pontuar várias amostras em paralelo (com limite de concorrência)
async def limited_get_score(dataset):
    async def gen(sample):
        async with sem:
            res = await get_eval_score(sample)
            progress_bar.update(1)
            return res

    progress_bar = tqdm_asyncio(total=len(dataset), desc="Pontuando", unit="amostra")
    tasks = [gen(sample) for sample in dataset]
    responses = await tqdm_asyncio.gather(*tasks)
    progress_bar.close()
    return responses

A última peça são os dados: um conjunto sintético de 10 pares de pergunta/contexto sobre o Brasil (economia, empresas e cidades), simulando o que um sistema de RAG teria recuperado.

In [48]:
eval_samples = [
    {
        "question": "Qual foi a inflação medida pelo IPCA em 2023?",
        "context": "O Índice Nacional de Preços ao Consumidor Amplo (IPCA), medido pelo IBGE, fechou 2023 em 4,62%, abaixo do teto da meta de inflação de 4,75%.",
    },
    {
        "question": "Quantas pessoas moram em São Paulo segundo o Censo 2022?",
        "context": "Segundo o Censo 2022 do IBGE, o município de São Paulo tem 11.451.245 habitantes, sendo a cidade mais populosa do Brasil e das Américas.",
    },
    {
        "question": "Qual foi a taxa Selic ao final de 2023?",
        "context": "Em dezembro de 2023, o Copom reduziu a taxa Selic para 11,75% ao ano, acumulando quatro cortes consecutivos de 0,5 ponto percentual.",
    },
    {
        "question": "Quantos aviões a Embraer entregou em 2023?",
        "context": "A Embraer entregou 181 aeronaves em 2023, entre jatos comerciais e executivos, um crescimento de 13% em relação às 160 entregas de 2022.",
    },
    {
        "question": "Qual foi a receita da Petrobras em 2023?",
        "context": "A Petrobras registrou receita líquida de US$ 102,4 bilhões em 2023, uma queda em relação aos US$ 124,5 bilhões de 2022, refletindo a redução dos preços do petróleo.",
    },
    {
        "question": "Quanto o Brasil produziu de café em 2023?",
        "context": "A produção brasileira de café em 2023 foi estimada pela Conab em 54,7 milhões de sacas de 60 kg, mantendo o Brasil como maior produtor mundial do grão.",
    },
    {
        "question": "Qual foi o crescimento do PIB brasileiro em 2023?",
        "context": "O Produto Interno Bruto (PIB) do Brasil cresceu 2,9% em 2023, segundo o IBGE, puxado principalmente pelo desempenho do agronegócio.",
    },
    {
        "question": "Quantas toneladas de minério de ferro a Vale produziu em 2023?",
        "context": "A Vale produziu 321 milhões de toneladas de minério de ferro em 2023, superando a meta anual e mantendo-se como uma das maiores mineradoras do mundo.",
    },
    {
        "question": "Qual foi o lucro do Itaú Unibanco em 2023?",
        "context": "O Itaú Unibanco registrou lucro líquido recorrente de R$ 35,6 bilhões em 2023, alta de 12,7% em relação ao ano anterior, o maior resultado entre os bancos brasileiros.",
    },
    {
        "question": "Qual é a população de Belo Horizonte?",
        "context": "Belo Horizonte, capital de Minas Gerais, tem 2.315.560 habitantes segundo o Censo 2022 do IBGE, sendo a sexta cidade mais populosa do Brasil.",
    },
]

print(f"Avaliação de {len(eval_samples)} amostras")
print(f"Exemplos few-shot: {len(few_shot_examples)}")

Avaliação de 10 amostras
Exemplos few-shot: 3


Primeiro, o Qwen3-8B gera as respostas do nosso "RAG" a partir dos contextos, em paralelo:

In [49]:
async def generate_answer(sample):
    prompt = f"""Use o contexto a seguir para responder à pergunta de forma direta e concisa.

Contexto: {sample["context"]}

Pergunta: {sample["question"]}"""
    response = await client.chat.completions.create(
        model=HF_MODEL_ID,  # modelo avaliado: Qwen/Qwen3-8B
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=2048,
    )
    return (response.choices[0].message.content or "").strip()

async def limited_generate(dataset):
    async def gen(sample):
        async with sem:
            answer = await generate_answer(sample)
            progress_bar.update(1)
            return {**sample, "answer": answer}

    progress_bar = tqdm_asyncio(total=len(dataset), desc="Gerando respostas", unit="amostra")
    tasks = [gen(sample) for sample in dataset]
    responses = await tqdm_asyncio.gather(*tasks)
    progress_bar.close()
    return responses

eval_ds = await limited_generate(eval_samples)

# Adiciona uma resposta propositalmente incorreta para vermos o juiz em ação
eval_ds.append({
    "question": "Qual foi a taxa Selic ao final de 2023?",
    "context": "Em dezembro de 2023, o Copom manteve a taxa Selic em 11,75% ao ano, após dois cortes consecutivos de 0,5 ponto percentual.",
    "answer": "A Selic terminou 2023 em 6,75% ao ano.",
})




























Gerando respostas: 100%|██████████| 10/10 [00:05<00:00,  1.91amostra/s]


Vamos testar a avaliação com um exemplo:

In [50]:
sample = eval_ds[:1]
print(f"Pergunta: {sample[0]['question']}\nContexto: {sample[0]['context']}\nResposta: {sample[0]['answer']}")
print("---" * 10)
# fora de um notebook jupyter, use:
# responses = asyncio.run(limited_get_score(sample))
responses = await limited_get_score(sample)
print(f"Raciocínio: {responses[0]['reasoning']}\nNota total: {responses[0]['total_score']}")

Pergunta: Qual foi a inflação medida pelo IPCA em 2023?
Contexto: O Índice Nacional de Preços ao Consumidor Amplo (IPCA), medido pelo IBGE, fechou 2023 em 4,62%, abaixo do teto da meta de inflação de 4,75%.
Resposta: A inflação medida pelo IPCA em 2023 foi de 4,62%.
------------------------------








Pontuando: 100%|██████████| 1/1 [00:04<00:00,  4.40s/amostra]

Raciocínio: 1. Contexto: a resposta usa apenas a informação do contexto (4,62%) e não inclui detalhes externos. 1 ponto. 2. Completude: responde exatamente o valor pedido (4,62%) com base no contexto. 1 ponto. 3. Concisão: a resposta é direta, sem redundâncias ou informações adicionais. 1 ponto.
Nota total: 3


Funcionou! Agora vamos avaliar todas as amostras e calcular a nota média.

In [51]:
results = await limited_get_score(eval_ds)




























Pontuando: 100%|██████████| 11/11 [00:14<00:00,  1.35s/amostra]


In [52]:
# calcula a nota média
total_score = sum([r["total_score"] for r in results]) / len(results)
print(f"Nota média: {total_score:.2f}")

# amostras com a menor nota
min_score = min(r["total_score"] for r in results)
worst = [r for r in results if r["total_score"] == min_score]
print(f"Menor nota: {min_score} ({len(worst)} amostras)")

Nota média: 2.73
Menor nota: 0 (1 amostras)


Para entender melhor o resultado, vamos analisar um exemplo com a menor nota e verificar se a avaliação faz sentido:

In [53]:
print(f"Pergunta: {worst[0]['question']}\nContexto: {worst[0]['context']}\nResposta: {worst[0]['answer']}")
print("---" * 10)
print(f"Raciocínio: {worst[0]['reasoning']}\nNota total: {worst[0]['total_score']}")

Pergunta: Qual foi a taxa Selic ao final de 2023?
Contexto: Em dezembro de 2023, o Copom manteve a taxa Selic em 11,75% ao ano, após dois cortes consecutivos de 0,5 ponto percentual.
Resposta: A Selic terminou 2023 em 6,75% ao ano.
------------------------------
Raciocínio: 1. Contexto: a resposta afirma que a Selic terminou em 6,75% ao ano, mas o contexto especifica que a taxa foi mantida em 11,75% ao ano. Isso contradiz diretamente o contexto, introduzindo uma informação incorreta. 0 pontos. 2. Completude: a resposta não fornece a informação correta (11,75%) e, portanto, não cobre o elemento-chave da pergunta. 0 pontos. 3. Concisão: a resposta é curta, mas sua concisão não compensa a inacuracidade. 0 pontos.
Nota total: 0


O juiz (Qwen3-14B) consegue identificar quando a resposta diverge do contexto ou deixa a desejar nos critérios. Note como completude e concisão dependem fortemente do contexto. Dependendo das suas necessidades, há espaço para refinar o prompt e os critérios da métrica.